# B5 — Validation-selected B2+B4 ensemble

This notebook reuses the authoritative B2 `NCSU_DRCNN` and B4 `CompactBNPool` checkpoints. It selects one of three pre-registered probability blends using only unseen-layout validation predictions. Frozen test predictions are created only if the validation gate passes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
B2_CHECKPOINTS = Path('/content/drive/MyDrive/ADVLSI2_B2/b2_baselines/checkpoints')
B4_CHECKPOINTS = Path('/content/drive/MyDrive/ADVLSI2_B4/b4_architecture/checkpoints')
OUTPUT_DIR = Path('/content/drive/MyDrive/ADVLSI2_B5/b5_ensemble')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert B2_CHECKPOINTS.is_dir(), f'Missing B2 checkpoints: {B2_CHECKPOINTS}'
assert B4_CHECKPOINTS.is_dir(), f'Missing B4 checkpoints: {B4_CHECKPOINTS}'
print(f'B2 checkpoints: {B2_CHECKPOINTS}')
print(f'B4 checkpoints: {B4_CHECKPOINTS}')
print(f'Persistent B5 results: {OUTPUT_DIR}')

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile

REPOSITORY = 'https://github.com/nocleo/ADVLSI2_Project_updated.git'
BRANCH = 'agent/b5-ensemble'
CHECKOUT = Path(tempfile.mkdtemp(prefix='advlsi-b5-')) / 'ADVLSI2_Project_updated'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPOSITORY, str(CHECKOUT)], check=True)
os.chdir(CHECKOUT)

import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before running B5.'
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
environment = os.environ.copy()
environment['PYTHONUNBUFFERED'] = '1'
command = [
    sys.executable, 'scripts/run_b5_ensemble.py',
    '--python', sys.executable,
    '--b2-checkpoints', str(B2_CHECKPOINTS),
    '--b4-checkpoints', str(B4_CHECKPOINTS),
    '--output-dir', str(OUTPUT_DIR),
]
print(' '.join(command))
subprocess.run(command, check=True, env=environment)

In [ ]:
from IPython.display import Markdown, display
report = OUTPUT_DIR / 'README.md'
if report.exists():
    display(Markdown(report.read_text()))
else:
    print('No report was written; inspect the preceding error and rerun to resume.')

In [ ]:
from google.colab import files
archive = shutil.make_archive('/content/ADVLSI2_B5_results', 'zip', OUTPUT_DIR)
files.download(archive)